## ELN risk group reclassification — Stage 1 of 3: Data generation

This notebook prepares input data for ELN risk group classification using the R script.

**Pipeline:**
1. **This notebook** — Merges model predictions with clinical data and generates one CSV per label-replacement mode into `data/{DATA_VERSION_IDENTIFIER}/eln/r_input/`. Also writes `eln_run_config.R` at the project root so the R script knows where to read from and write to.
2. **`7_modify_eln_risk_groups.R`** — Run manually after this notebook. Applies ELN 2022 classification logic and writes results to `r_output/`.
3. **`6_eln_risk_group_analysis.ipynb`** — Reads R output and produces Kaplan-Meier survival curves and Sankey patient-flow diagrams.

In [2]:
from pathlib import Path
from config import (
    USE_DUMMY_DATA,
    BM_RUN_PATH, PB_RUN_PATH, FUSION_RUN_PATH,
    CLINICAL_DATA_PATH_BM, SURVIVAL_DATA_PATH_BM, VAF_DATA_PATH_BM,
    SELECTED_VARIABLES,
    DATA_VERSION_IDENTIFIER,
)


if False:
    SELECTED_VARIABLES = [
        "CBFB-MYH11",
        "IDH2",
        "IDH_mutations",
        "KIT",
        "NPM1",
        "PML-RARA",
        "RTK_RAS_mutations",
        "STAG2",
        "TP53",
        "monosomy5_del_5q",
    ]

MUTATION_LABEL_RENAME = {
    "RUNX1-RUNX1T1_OR_t_8_21": "RUNX1-RUNX1T1",
}

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_PREDICTIONS_PATH_BM     = BM_RUN_PATH
MODEL_PREDICTIONS_PATH_PB     = PB_RUN_PATH
MODEL_PREDICTIONS_PATH_FUSION = FUSION_RUN_PATH
CLINICAL_DATA_FILE = CLINICAL_DATA_PATH_BM
SURVIVAL_DATA_FILE = SURVIVAL_DATA_PATH_BM
VAF_DATA_FILE      = VAF_DATA_PATH_BM

# Base directory for ELN R script I/O.
# Default: local data folder. Override when running in the academic environment.
ELN_BASE_DIR = Path(f"data/{DATA_VERSION_IDENTIFIER}/eln")
# ELN_BASE_DIR = Path(f"/home/HUSTIETOALLAS/ext11670944/mounts/research/husdatalake/disease/scripts/AML_mutation_prediction/aml-mutation-prediction/data/{DATA_VERSION_IDENTIFIER}/eln")

# Label for the clinical data used above ("bm" or "pb") — included in output filenames.
# For FUSION: set this to whichever clinical data table you chose above.
SLIDE_TYPE = "bm"

MODES = ["TARGETS", "TP", "FN", "TP_FP", "TP_FP_FN"]
# these did not lead to improvement in risk stratification: ["FN", "TP_FP", "TP_FP_FN"]

VARIABLE_SETS = {
    "selected_variables": SELECTED_VARIABLES,
    "NPM1": ["NPM1"],
}
# ──────────────────────────────────────────────────────────────────────────────

### Label replacement modes

Each mode controls how the true mutation labels are replaced before ELN risk groups are recomputed. Using NPM1 as a concrete example (NPM1-positive → ELN Favorable):

| Mode | What happens to labels | Clinical question |
|---|---|---|
| **TARGETS** | Original clinical labels unchanged | Baseline: how well do current lab-confirmed labels stratify survival? |
| **TP** | FN patients flipped to negative; TP patients kept positive | If we only trust mutations the model *confirmed*, does ELN stratification improve? |
| **FN** | TP patients flipped to negative; FN patients kept positive | Are the cases the model *missed* clinically distinct? (inverted test) |
| **TP_FP** | Like TP, plus FP patients elevated to positive | What if we fully trust the model — including cases it predicts positive but clinical labs called negative? |
| **TP_FP_FN** | Same as TP_FP (effectively identical in the code) | Intended as a combined mode; the extra FN mask is redundant with the TP step |

The key comparison is **TARGETS vs TP**: if the model's predictions carry real biological signal, the TP mode should produce KM curves with better separation between Favorable/Intermediate/Adverse — because FN patients (who the model couldn't detect as mutation-positive) presumably have a different prognosis than the true TPs, and TP mode moves them out of the Favorable group.

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from core.utils import load_predictions, add_confusion_categories


In [4]:
def get_data(model_results_path, clinical_data_path, survival_data_path, selected_variables):
    predicted_labels = load_predictions(model_results_path, selected_variables)
    predicted_labels = add_confusion_categories(predicted_labels, selected_variables)


    clinical_data = pd.read_csv(clinical_data_path)
    clinical_data.album_id = clinical_data.album_id.astype(str)
    predicted_labels.album_id = predicted_labels.album_id.astype(str)
    clinical_data = clinical_data.rename(columns={"year": "database_id"})

    df_merged = pd.merge(clinical_data, predicted_labels, on=["album_id", "database_id"], how="inner")

    df_survival = pd.read_excel(survival_data_path)
    eln_analysis_patients = df_survival[df_survival.cyto == True].henkilotunnus.to_list()
    df_merged = df_merged[df_merged.henkilotunnus.isin(eln_analysis_patients)]
    return df_merged


def add_confusion_categories(df, target_variables=None, keep_original=False):
    if target_variables is None:
        target_variables = [col[:-len("_true_label")] for col in df.columns if col.endswith("_true_label")]

    df = df.copy()
    for variable in target_variables:
        true_col = f"{variable}_true_label"
        pred_col = f"{variable}_pred_label"
        true = df[true_col]
        pred = df[pred_col]
        has_label = ~true.isna()
        category = pd.Series(index=df.index, dtype=object)
        category[has_label & (true == 1) & (pred == 1)] = "TP"
        category[has_label & (true == 0) & (pred == 0)] = "TN"
        category[has_label & (true == 0) & (pred == 1)] = "FP"
        category[has_label & (true == 1) & (pred == 0)] = "FN"
        df[f"{variable}_predicted"] = category

    if not keep_original:
        drop_cols = [
            col for variable in target_variables
            for col in (f"{variable}_true_label", f"{variable}_pred_label", f"{variable}_pred_prob")
            if col in df.columns
        ]
        df = df.drop(columns=drop_cols)

    return df

In [5]:
def replace_target_labels_with_predicted(df, columns, mode):
    stats = []
    any_changed_mask = pd.Series(False, index=df.index)

    for col in columns:
        
        pred_col = f"{col}_predicted"
        try:
            original = df[col]
        except KeyError:
            if col in MUTATION_LABEL_RENAME.keys():
                original = df[MUTATION_LABEL_RENAME.get(col)]
            else:
                print(f"Skipping variable: {col}")
            
        predicted = df[pred_col]

        unique_vals = original.dropna().unique()
        if set(unique_vals).issubset({"POS", "NEG"}):
            POS_VALUE, NEG_VALUE = "POS", "NEG"
        elif set(unique_vals).issubset({0, 1}):
            POS_VALUE, NEG_VALUE = 1, 0
        else:
            raise ValueError(f"Column {col} contains unsupported values: {unique_vals}")

        result = original.copy()

        if mode == "TARGETS":
            df = df.drop(columns=pred_col)
            continue

        if mode == "FN":
            result = result.mask((original == POS_VALUE) & (predicted == "TP"), NEG_VALUE)
        else:
            result = result.mask((original == POS_VALUE) & (predicted != "TP"), NEG_VALUE)

        if mode in ("TP_FP", "TP_FP_FN"):
            result = result.mask((original == NEG_VALUE) & (predicted == "FP"), POS_VALUE)

        if mode == "TP_FP_FN":
            result = result.mask((original == POS_VALUE) & (predicted == "FN"), NEG_VALUE)

        df[col] = result
        df = df.drop(columns=pred_col)

        changed_mask = (original != result) & ~(original.isna() & result.isna())
        any_changed_mask |= changed_mask

        stats.append({
            "mutation": col,
            "POS_to_NEG": ((original == POS_VALUE) & (result == NEG_VALUE)).sum(),
            "NEG_to_POS": ((original == NEG_VALUE) & (result == POS_VALUE)).sum(),
            "total_POS_before": (original == POS_VALUE).sum(),
            "total_NEG_before": (original == NEG_VALUE).sum(),
            "total_POS_after": (result == POS_VALUE).sum(),
            "total_NEG_after": (result == NEG_VALUE).sum(),
        })

    return df, pd.DataFrame(stats), df.loc[any_changed_mask, "henkilotunnus"].to_list()


def modify_target_labels_for_eln_stratification(df_input, r_input_dir, slide_type, run_id, experiments, variables, model_type="fusion", display_stats=False):
    modification_stats, modified_patients = {}, {}
    for variable_set_name, variable_set in variables.items():
        modification_stats[variable_set_name] = {}
        modified_patients[variable_set_name] = {}
        for experiment in experiments:
            df, stats, patients = replace_target_labels_with_predicted(df_input.copy(), columns=variable_set, mode=experiment)
            filename = f"aml_mut_pred_{experiment}_{variable_set_name}_{model_type}_clinical_data_{slide_type}_{run_id}.csv"
            df.to_csv(Path(r_input_dir) / filename, index=False)
            modification_stats[variable_set_name][experiment] = stats
            modified_patients[variable_set_name][experiment] = patients
            if display_stats:
                print(f"{variable_set_name} / {experiment}")
                display(stats)
    return modification_stats, modified_patients

In [6]:
R_INPUT_DIR  = ELN_BASE_DIR / "r_input"
R_OUTPUT_DIR = ELN_BASE_DIR / "r_output"

r_input_ready  = R_INPUT_DIR.exists()  and any(R_INPUT_DIR.iterdir())
r_output_ready = R_OUTPUT_DIR.exists() and any(R_OUTPUT_DIR.iterdir())

if r_input_ready:
    print(f"Input data already generated for '{DATA_VERSION_IDENTIFIER}'.")
    print("  → To regenerate: delete the r_input folder or set a new DATA_VERSION_IDENTIFIER.")
else:
    print(f"r_input is empty — ready to generate.")

if r_output_ready:
    print(f"R output already exists for '{DATA_VERSION_IDENTIFIER}' — no need to run R script.")
    print("  → Proceed directly to the analysis notebook.")
else:
    print(f"r_output is empty — run R after generating input data.")

R_INPUT_DIR.mkdir(parents=True, exist_ok=True)
R_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

r_config = Path("eln_run_config.R")   # project root — always overwritten on new run
r_config.write_text(
    f'INPUT_DATA_DIR  <- "{R_INPUT_DIR.resolve()}"\n'
    f'OUTPUT_DATA_DIR <- "{R_OUTPUT_DIR.resolve()}"\n'
)
print(f"\nR config written to: {r_config.resolve()}")
print(f"  INPUT_DATA_DIR  = {R_INPUT_DIR.resolve()}")
print(f"  OUTPUT_DATA_DIR = {R_OUTPUT_DIR.resolve()}")

r_input is empty — ready to generate.
r_output is empty — run R after generating input data.

R config written to: /home/HUSTIETOALLAS/ext11670944/mounts/research/husdatalake/disease/scripts/AML_mutation_prediction/aml_mut_pred_analysis/eln_run_config.R
  INPUT_DATA_DIR  = /home/HUSTIETOALLAS/ext11670944/mounts/research/husdatalake/disease/scripts/AML_mutation_prediction/aml_mut_pred_analysis/data/03-06-2026/eln/r_input
  OUTPUT_DATA_DIR = /home/HUSTIETOALLAS/ext11670944/mounts/research/husdatalake/disease/scripts/AML_mutation_prediction/aml_mut_pred_analysis/data/03-06-2026/eln/r_output


In [7]:
if not r_input_ready:
    df_fusion = get_data(MODEL_PREDICTIONS_PATH_FUSION, CLINICAL_DATA_FILE, SURVIVAL_DATA_FILE, SELECTED_VARIABLES)
    stats_fusion, _ = modify_target_labels_for_eln_stratification(
        df_fusion, R_INPUT_DIR, SLIDE_TYPE, DATA_VERSION_IDENTIFIER, MODES, VARIABLE_SETS, model_type="fusion"
    )
    df_bm = get_data(MODEL_PREDICTIONS_PATH_BM, CLINICAL_DATA_FILE, SURVIVAL_DATA_FILE, SELECTED_VARIABLES)
    stats_bm, _ = modify_target_labels_for_eln_stratification(
        df_bm, R_INPUT_DIR, SLIDE_TYPE, DATA_VERSION_IDENTIFIER, MODES, VARIABLE_SETS, model_type="bm"
    )
    df_pb = get_data(MODEL_PREDICTIONS_PATH_PB, CLINICAL_DATA_FILE, SURVIVAL_DATA_FILE, SELECTED_VARIABLES)
    stats_pb, _ = modify_target_labels_for_eln_stratification(
        df_pb, R_INPUT_DIR, SLIDE_TYPE, DATA_VERSION_IDENTIFIER, MODES, VARIABLE_SETS, model_type="pb"
    )
else:
    print("Skipped — input data already exists.")

Skipping variable: NPM1_AND_FLT3_ITD
Skipping variable: NPM1_AND_FLT3_ITD
Skipping variable: NPM1_AND_FLT3_ITD
Skipping variable: NPM1_AND_FLT3_ITD
Skipping variable: NPM1_AND_FLT3_ITD


### Next step: Run the R script

```bash
Rscript 7_modify_eln_risk_groups.R
```

Launch R-studio: `mounts/research/src/launch/launch_rstudio.sh`

Then open `6_eln_risk_group_analysis.ipynb` to run the analysis.